# Introductory digital pathology demo
## Goal · 15–20 minutes
An introductory demo using a **locally available H&E lung slide** from TCGA-LUAD (case **TCGA-55-6972**).

By the end, you should be able to inspect a WSI pyramid, explain **microns per pixel (MPP)**, create and inspect a tissue mask, extract a small set of calibrated tiles, and save QC decisions with coordinates.

**Workflow:** inspect → mask → sample tiles → review quality → export.

This is a teaching workflow, not a validated diagnostic pipeline. A tissue mask does not identify tumour, and a patient-level diagnosis is not a label for every tile.

**Suggested pace:** inspection 3 min; combined masking and parameters 6 min; tiling 4 min; QC 3 min; export and discussion 3 min.

## Setup
Open this notebook from the cloned project directory or its `notebooks/` folder. In Jupyter or VS Code, select this project's `.venv` Python interpreter from the locked `uv` environment.

Then **Restart Kernel and Run All**. No downloads, model weights or GPU are needed once the demo slide is available locally. The first cell discovers the project from the working directory; set its optional `PROJECT` value if you launch the notebook elsewhere. The source WSI is read only, and each run creates a new output folder.

We use the installed OpenSlide, TIAToolbox, NumPy, pandas, Pillow, OpenCV, OmegaConf and Matplotlib packages. The notebook calls reusable stage functions in `src/`; the Hydra CLI is not invoked. `inspection.py`, `masking.py`, `tiling.py` and `quality.py` mirror the steps below.

The version in Git omits embedded outputs to keep it small and reviewable. Running it displays the examples and saves figures, tiles and provenance under `outputs/introductory_demo/<run_id>/`. Save executed notebook copies under `outputs/` as well; that directory is excluded from Git.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from copy import deepcopy
from omegaconf import OmegaConf
import hashlib
import json
import platform
import time
import sys
import uuid

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import openslide
import pandas as pd
from PIL import Image
from IPython.display import display, HTML

PROJECT = None  # Optional: set Path(...) to the checkout when launching elsewhere.
if PROJECT is None:
    PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').is_file()
                    and (p / 'src/preprocess.py').is_file()), None)
    if PROJECT is None:
        raise FileNotFoundError('Open this notebook from the project, or set PROJECT to its checkout directory.')
PROJECT = Path(PROJECT).expanduser().resolve(strict=True)
assert (PROJECT / 'src/preprocess.py').is_file(), 'PROJECT must point to this project checkout.'
sys.path.insert(0, str(PROJECT))
from src.inspection import read_thumbnail, read_slide_metadata
from src.image_utils import save_png, mean_background_color
from src.pipeline import preprocess_masks
from src.filters import resolve_pen_thresholds
from src.tiling import calibrated_patch_size, clam_coordinates, read_tile, open_clam_slide, clam_contours, contour_tile_mask, CLAM_COMMIT
from src.quality import tile_quality
from src.normalization import prepare_stain_image, compare_stain_normalization
SLIDE_ID = '20c9afac-0d0c-4fcf-b3bf-a2beba3dd76b'
SLIDE_PATH = PROJECT / 'data/tcga/TCGA-LUAD' / f'{SLIDE_ID}.svs'
OUTPUT_ROOT = PROJECT / 'outputs' / 'introductory_demo'
THUMBNAIL_EDGE = 1024
MAX_THUMBNAIL_READ_PIXELS = 4_000_000
TARGET_MPP = 0.5                 # micrometres per output pixel
TILE_PIXELS = 256                # output width and height
MIN_TISSUE_FRACTION = 0.60       # illustrative, not a universal cutoff
N_TISSUE_CANDIDATES = 24
N_LOW_TISSUE_CONTROLS = 12
CLAM_CONTOUR_FN = "four_pt"     # four_pt_hard, center or basic are alternatives
MAX_CONTOURS = 128
MAX_GRID_POSITIONS = 20_000
MAX_TILE_READ_PIXELS = 1_000_000
MAX_TILE_SECONDS = 120           # checked between reads; each read is pixel-bounded
stain_provenance = None          # Export remains valid if optional normalisation is skipped.

assert 0 < TARGET_MPP and 64 <= TILE_PIXELS <= 512
assert 128 <= THUMBNAIL_EDGE <= 2048
assert 0 <= MIN_TISSUE_FRACTION <= 1
assert 0 <= N_TISSUE_CANDIDATES + N_LOW_TISSUE_CONTROLS <= 50
assert SLIDE_PATH.is_file(), f'Local slide missing: {SLIDE_PATH}'
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12})
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = OUTPUT_ROOT / run_id
OUT.mkdir(parents=True, exist_ok=False)
started = time.monotonic()
print(f'Python: {sys.executable}')
print(f'Outputs: {OUT}')

### Change the preprocessing parameters here
These settings load the repository defaults. Edit the assignments, then **Restart Kernel and Run All**. Sizes and areas below are in thumbnail pixels; changing `THUMBNAIL_EDGE` changes their physical meaning.

| Control | Effect |
| --- | --- |
| `ARTIFACTS['pen_colors']` | Enabled pen colours; `[]` disables pen masking |
| `ARTIFACTS['pen_thresholds']` | Per-colour replacement rule lists; `None` uses CODAIT defaults |
| `MASKING['tia_threshold']` | `None` fits grayscale Otsu; a number overrides it (higher retains lighter pixels) |
| `MASKING['tia_enabled']` | Enable/disable the TIA preliminary gate |
| `MASKING['median_size']` | Odd saturation smoothing window, 1–31 |
| `MASKING['use_otsu']`, `['saturation_threshold']` | Automatic CLAM saturation threshold, or fixed 0–255 threshold |
| `MASKING['closing_size']` | Morphological closing window, 0–31; 0 disables |
| `MASKING['max_hole_area']` | Largest enclosed hole to fill; 0 preserves all remaining holes |
| `ARTIFACTS['min_tissue_area']` | Smallest retained tissue component |

**Custom pen rules:** each `[r, g, b]` triple is one rule. Red means `R>r & G<g & B<b`; green means `R<r & G>g & B>b`; blue means `R<r & G<g & B>b`. Values are integers 0–255. Rules within a colour are OR'ed: tightening one rule may have no effect if another still matches. Supplying a colour replaces its entire list; omitted colours keep their defaults. These are colour heuristics and can remove real stained tissue.


In [ ]:
MASKING = OmegaConf.to_container(OmegaConf.load(PROJECT / 'configs/masking/codait_tia_clam.yaml'), resolve=True)
ARTIFACTS = OmegaConf.to_container(OmegaConf.load(PROJECT / 'configs/artifacts/default.yaml'), resolve=True)

# Editable controls: these values reproduce the current repository default.
MASKING.update(tia_enabled=True, tia_threshold=None, median_size=7,
               use_otsu=True, saturation_threshold=20, closing_size=3, max_hole_area=16)
ARTIFACTS.update(pen_colors=['red', 'green', 'blue'], pen_thresholds=None,
                 remove_dark=True, dark_threshold=30, remove_gray=False,
                 gray_tolerance=15, min_tissue_area=64)

# Example: replace ALL blue rules with one stricter blue rule:
# ARTIFACTS['pen_thresholds'] = {'blue': [[60, 100, 210]]}
# Example: edit one original rule and preserve the other blue rules:
# custom = {color: [list(rule) for rule in rules]
#           for color, rules in resolve_pen_thresholds().items()}
# custom['blue'][0] = [60, 100, 210]
# ARTIFACTS['pen_thresholds'] = custom
# Example: use a fixed CLAM threshold instead of saturation Otsu:
# MASKING.update(use_otsu=False, saturation_threshold=30)
print(OmegaConf.to_yaml(OmegaConf.create({'masking': MASKING, 'artifacts': ARTIFACTS})))


## Steps
### 1. Inspect the slide and its physical scale
A WSI is usually stored as an **image pyramid**. Level 0 has the most detail; higher-numbered levels are smaller. We read a small pyramid level for the overview and local regions for individual tiles.

**MPP** tells us how much tissue each pixel represents. At 0.5 µm/pixel, a 256-pixel tile covers **128 × 128 µm**. A “20×” label alone is less precise than physical calibration.

The local manifest records this public diagnostic slide and its expected checksum. Checking the checksum identifies the exact source without copying the WSI.

In [ ]:
manifest_path = PROJECT / 'manifests/tcga_demo.csv'
manifest = pd.read_csv(manifest_path)
source = manifest.loc[manifest.file_id.eq(SLIDE_ID)]
assert len(source) == 1, 'Expected exactly one source record'
source = source.iloc[0].to_dict()
with SLIDE_PATH.open('rb') as stream:
    source_md5 = hashlib.file_digest(stream, 'md5').hexdigest()
assert source_md5 == source['md5sum'], 'WSI differs from the local manifest'

slide_metadata = read_slide_metadata(SLIDE_PATH)
width, height = slide_metadata['dimensions']
levels = slide_metadata['level_dimensions']
downsamples = slide_metadata['level_downsamples']
mpp_x, mpp_y = slide_metadata['native_mpp_xy']
assert mpp_x is not None and mpp_y is not None, 'Recorded MPP is required; do not infer it from objective power.'
assert np.isfinite([mpp_x, mpp_y]).all() and min(mpp_x, mpp_y) > 0
(OUT / 'wsi_metadata.json').write_text(json.dumps(slide_metadata, indent=2))

summary = {key: slide_metadata[key] for key in
           ['filename', 'file_size_bytes', 'vendor', 'objective_power', 'native_mpp_xy',
            'dimensions', 'physical_size_mm', 'level_count', 'bounds_level0',
            'associated_images', 'icc_profile_present', 'icc_profile_description']}
display(pd.DataFrame({'field': list(summary), 'value': [str(v) for v in summary.values()]}))
display(pd.DataFrame({'level': range(len(levels)), 'width_px': [v[0] for v in levels],
                      'height_px': [v[1] for v in levels], 'downsample': downsamples,
                      'approx_mpp_x': np.asarray(downsamples) * mpp_x,
                      'approx_mpp_y': np.asarray(downsamples) * mpp_y}))
raw_properties = pd.DataFrame(sorted(slide_metadata['properties'].items()), columns=['OpenSlide property', 'value'])
display(HTML('<details><summary>Expand all raw scanner/OpenSlide properties</summary>'
             + raw_properties.to_html(index=False, escape=True) + '</details>'))
print(f"Verified source checksum; read {len(raw_properties)} properties without decoding slide pixels.")


### Now read the first slide image
Metadata came first: no WSI or associated-image pixels were decoded in the previous cell. Scanner properties can be missing or vendor-specific. Physical slide size includes background and is not specimen size. Associated-image **names** are listed, but label/macro images are not displayed.

We now read a bounded low-resolution pyramid level for the overview. The full gigapixel image stays on disk.

In [ ]:
rgb, inspection = read_thumbnail(SLIDE_PATH, THUMBNAIL_EDGE,
                                 MAX_THUMBNAIL_READ_PIXELS, MAX_THUMBNAIL_READ_PIXELS)
thumbnail = Image.fromarray(rgb)
thumb_level = inspection['read_level']
thumb_h, thumb_w = rgb.shape[:2]
scale_x, scale_y = width / thumb_w, height / thumb_h
fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(rgb)
ax.set_title('Local H&E WSI · low-resolution overview')
ax.axis('off')
fig.tight_layout()
fig.savefig(OUT / '01_overview.png')
plt.show()

**Look together:** identify coloured tissue and background; inspect high-resolution tiles later for folds, pen marks, bubbles and blur. A thumbnail cannot establish focus quality. No tumour regions are annotated here. Colours are decoded scanner RGB; ICC conversion is not applied.

### 2. Run the default combined tissue/QC pipeline
The notebook and CLI now share `src.pipeline.preprocess_masks`:

1. **CODAIT colour rules** identify pen-like pixels, plus configured dark/grey pixels. Replace these with white in a working image, leaving source RGB intact.
2. **TIA Otsu** finds a grayscale threshold on that working image. Whiten pixels outside its preliminary tissue mask.
3. **CLAM-style masking** applies HSV saturation → median filter → fixed or Otsu saturation threshold → optional closing. This is a second, separate threshold in a different colour channel.
4. **Small-hole cleanup** fills only enclosed background components up to the chosen area. Larger holes and border-connected spaces remain; closing can also bridge spaces, so inspect both settings.
5. **Final QC** reapplies CODAIT exclusions and removes small tissue components. Hole filling cannot bring pen-excluded pixels back.

Our hole policy uses **raster areas in thumbnail pixels**, not upstream CLAM's reference-patch-scaled contour areas or maximum-hole count. It is CLAM-style preprocessing, not numerical equivalence to CLAM. Filling holes can erase real lung air spaces: compare the intermediate masks before choosing parameters.


In [ ]:
result = preprocess_masks(rgb, masking=MASKING, artifacts=ARTIFACTS)
mask = result['clean']
otsu_threshold = result['metadata']['clam_saturation_threshold']
assert mask.any(), 'No tissue retained: inspect the intermediate masks and parameters.'
assert not np.any(mask & result['excluded'])
overlay = rgb.copy()
overlay[mask] = (0.65 * rgb[mask] + 0.35 * np.array([20, 150, 185])).astype(np.uint8)
views = [rgb, result['stages']['codait_filtered'], result['stages']['tia_otsu'],
         result['stages']['clam_mask'], result['stages']['filled_holes'], overlay]
titles = ['Original', '1 · After CODAIT', '2 · TIA grayscale gate',
          '3 · CLAM-style mask', '4 · Filled small holes', '5 · Final retained overlay']
fig, axes = plt.subplots(2, 3, figsize=(12, 10), layout='constrained')
for ax, im, title in zip(axes.flat, views, titles):
    ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
fig.savefig(OUT / '02_tissue_mask.png')
plt.show()
tia_threshold = result['metadata']['tia_threshold']
tia_threshold_text = 'disabled' if tia_threshold is None else f'{tia_threshold:.0f}/255'
print(f"TIA grayscale threshold: {tia_threshold_text}; "
      f"CLAM saturation threshold: {otsu_threshold:.0f}/255")
print(f"Filled {result['metadata']['filled_hole_pixels']} thumbnail pixels; "
      f"final retained area {mask.mean():.1%} (not a tumour percentage).")


### Parameter experiment: change one step at a time
The following comparisons use the same real thumbnail and do **not** replace the main `mask` used for tiling. One variant disables small-hole filling; the other replaces the blue pen rules. A zero difference means that setting did not affect this slide under the other active filters—not that it never matters.


In [ ]:
no_holes = {**MASKING, 'max_hole_area': 0}
custom_pen = deepcopy(ARTIFACTS)
custom_pen['pen_thresholds'] = deepcopy(ARTIFACTS.get('pen_thresholds') or {})
custom_pen['pen_thresholds']['blue'] = [[60, 100, 210]]
variants = [('Active settings', result),
            ('No small-hole filling', preprocess_masks(rgb, masking=no_holes, artifacts=ARTIFACTS)),
            ('Custom blue pen rules', preprocess_masks(rgb, masking=MASKING, artifacts=custom_pen))]
fig, axes = plt.subplots(1, 3, figsize=(12, 5), layout='constrained')
comparison = []
for ax, (name, variant) in zip(axes, variants):
    ax.imshow(variant['clean'], cmap='gray')
    ax.set_title(name)
    ax.axis('off')
    comparison.append({'variant': name, 'retained_area_pct': 100 * variant['clean'].mean(),
                       'changed_mask_pixels': int(np.count_nonzero(mask != variant['clean'])),
                       'pen_candidate_pixels': sum(int(v.sum()) for k, v in variant['reasons'].items()
                                                   if k.startswith('pen_'))})
fig.savefig(OUT / '02b_parameter_comparison.png')
plt.show()
display(pd.DataFrame(comparison).round(2))
print('Pen candidate counts sum per-colour masks; overlaps may be counted more than once.')


### 3. Choose tiles in physical units with upstream CLAM
The final binary mask is converted into tissue contours and child-hole contours in level-0 coordinates. We call **CLAM `WholeSlideImage.process_contour`** (pinned revision shown below), with padding disabled. The default `four_pt` rule accepts a tile if at least one of four interior points lies in a tissue contour; tile centres inside hole contours are excluded. `four_pt_hard` requires all four points. CLAM creates a grid starting at each contour's bounding-box origin, not the slide origin.

The 256-pixel output tiles span approximately 128 µm at 0.5 MPP. We read **up to 24 CLAM tiles with sufficient mask coverage and 12 lower-coverage CLAM tiles for QC comparison**. CLAM contour acceptance does not imply a tile is entirely tissue. This is a teaching sample, not a representative quality estimate.

Source reads use **CLAM's OpenSlide handle**, not TIAToolbox. CLAM uses at most four coordinate workers per contour; source-pixel reads remain sequential. Candidate and contour budgets are checked before tiling. Locations use level-0 `(x, y)` while arrays index `[y, x]`.

In [ ]:
(read_w, read_h), (actual_mpp_x, actual_mpp_y) = calibrated_patch_size(
    TILE_PIXELS, TARGET_MPP, (mpp_x, mpp_y))
assert read_w * read_h <= MAX_TILE_READ_PIXELS
assert read_w == read_h, 'CLAM requires square source patches.'
clam_slide = open_clam_slide(SLIDE_PATH)
try:
    coords, fractions, _ = clam_coordinates(clam_slide, mask, read_w, min_fraction=0.0,
        max_candidates=MAX_GRID_POSITIONS, max_contours=MAX_CONTOURS, contour_fn=CLAM_CONTOUR_FN)
finally:
    clam_slide.wsi.close()
grid = pd.DataFrame(coords, columns=['x_level0', 'y_level0'])
grid['thumbnail_tissue_fraction'] = fractions
print('Upstream CLAM revision:', CLAM_COMMIT)
assert not grid.empty, 'Slide is smaller than one tile.'

def evenly_spaced(frame, n):
    if frame.empty or n == 0:
        return frame.iloc[:0]
    return frame.iloc[np.linspace(0, len(frame) - 1, min(n, len(frame)), dtype=int)]

candidate = grid.thumbnail_tissue_fraction >= MIN_TISSUE_FRACTION
selected = pd.concat([
    evenly_spaced(grid[candidate], N_TISSUE_CANDIDATES).assign(sample_group='tissue candidate'),
    evenly_spaced(grid[~candidate], N_LOW_TISSUE_CONTROLS).assign(sample_group='low-coverage control')
], ignore_index=True)
selected.insert(0, 'tile_id', [f'tile_{i:02d}' for i in range(len(selected))])
print(f'{len(grid):,} CLAM contour-selected positions; {int(candidate.sum()):,} tissue candidates; {len(selected)} reads selected.')
print(f'Source region: {read_w} × {read_h} px → {TILE_PIXELS} × {TILE_PIXELS} px')
print(f'Actual output MPP: {actual_mpp_x:.5f} × {actual_mpp_y:.5f}')

fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(rgb)
for row in selected.itertuples():
    is_candidate = row.sample_group == 'tissue candidate'
    ax.add_patch(Rectangle((row.x_level0 / scale_x, row.y_level0 / scale_y),
                           read_w / scale_x, read_h / scale_y, fill=False,
                           edgecolor='#0072B2' if is_candidate else '#D55E00',
                           linestyle='-' if is_candidate else '--', linewidth=1.2))
ax.set_title('Selected tiles: solid blue = candidate; dashed orange = control')
ax.axis('off')
fig.tight_layout()
fig.savefig(OUT / '03_tile_locations.png')
plt.show()

### 4. Read tiles and measure quality
We report two different quantities: **retained footprint coverage** from the combined thumbnail mask, and **coloured-pixel fraction** measured directly on each tile with the fitted CLAM saturation threshold. They differ because masking, closing, hole cleanup and downsampling alter the thumbnail representation.

For this demo a tile passes only if **both fractions** meet `MIN_TISSUE_FRACTION`. Low-coverage controls cannot re-enter through the second rule. This is still not complete artefact QC: a rectangular tile can contain some excluded pixels even when it passes.

**Laplacian variance** is an edge/detail score for visual review, not a calibrated blur detector. It varies with tissue, scale and contrast. “Keep” means passes these coverage rules, not approved for analysis.


In [ ]:
tiles = {}
qc_rows = []
tile_started = time.monotonic()
reader = open_clam_slide(SLIDE_PATH)
try:
    for row in selected.itertuples():
        if time.monotonic() - tile_started > MAX_TILE_SECONDS:
            raise TimeoutError('Tile-reading budget exceeded; reduce the sample count.')
        tile_rgb = read_tile(reader, (row.x_level0, row.y_level0), (read_w, read_h),
                             TILE_PIXELS, max_read_pixels=MAX_TILE_READ_PIXELS)
        tile = Image.fromarray(tile_rgb)
        measures = tile_quality(tile_rgb, saturation_threshold=otsu_threshold)
        tissue_fraction = measures['tissue_fraction']
        detail_score = measures['laplacian_variance']
        passes = (row.thumbnail_tissue_fraction >= MIN_TISSUE_FRACTION
                  and tissue_fraction >= MIN_TISSUE_FRACTION)
        tiles[row.tile_id] = tile
        qc_rows.append({'tile_id': row.tile_id, 'tissue_fraction': tissue_fraction,
                        'laplacian_variance': detail_score,
                        'decision': 'keep' if passes else 'exclude',
                        'reason': 'coverage_pass_only' if passes
                                  else 'low_tissue_coverage'})
finally:
    handle = getattr(reader, 'wsi', None)
    if handle is not None:
        handle.close()
qc = selected.merge(pd.DataFrame(qc_rows), on='tile_id', validate='one_to_one')
display(qc[['tile_id', 'sample_group', 'tissue_fraction', 'laplacian_variance', 'decision']]
        .head(8).round(3))
print(qc.decision.value_counts().to_string())

### 5. Inspect accepted and rejected examples
Each row shows up to six evenly spaced examples from its decision group. Check both groups: a filter can exclude useful tissue or retain artefacts. Titles show **raw coloured-pixel percentage** and **detail score**.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 6.6), layout='constrained')
for ax in axes.flat:
    ax.axis('off')
for row_index, decision in enumerate(['keep', 'exclude']):
    examples = evenly_spaced(qc[qc.decision.eq(decision)], 6)
    for ax, row in zip(axes[row_index], examples.itertuples()):
        ax.imshow(tiles[row.tile_id])
        ax.set_title(f'{row.tile_id} · {decision}\n{row.tissue_fraction:.0%} coloured | detail {row.laplacian_variance:.0f}',
                     fontsize=10)
    if examples.empty:
        axes[row_index, 0].text(0.05, 0.5, f'No {decision} tiles', transform=axes[row_index, 0].transAxes)
fig.suptitle('Coverage decisions need visual review · each tile spans about 128 µm')
fig.savefig(OUT / '04_qc_contact_sheet.png')
plt.show()

kept = qc[qc.decision.eq('keep')].sort_values('laplacian_variance')
if not kept.empty:
    print('Lowest-detail retained tiles to inspect:', ', '.join(kept.head(3).tile_id))

### Optional: visualise TIA stain normalisation
Skip the next code cell to export the core workflow without stain comparisons.
We compare four installed TIA methods:

| Method | Main idea | What to inspect |
| --- | --- | --- |
| Reinhard | Match colour moments in LAB space | Sensitivity to tissue/background proportion |
| Macenko | Estimate stain directions in optical-density space | Estimated colours and concentration scaling |
| Ruifrok–Johnston | Use a fixed H&E stain matrix, then match concentrations | Whether fixed directions suit the stain/scanner |
| Vahadane | Estimate stain directions through dictionary learning | Stability and unexpected colour changes |

**Implementation caveat:** TIA 2.1.3 explicitly warns that its Vahadane implementation is unstable with newer scikit-learn dictionary learning. We include it as an exploratory teaching comparison, retain the warning, and set a reproducible NumPy seed before each fit/transform. Successful execution is not evidence that this implementation is scientifically reliable.

 We fit each TIA normaliser to one retained reference tile, then transform three other real tiles. The reference is the retained tile with the highest raw coloured-pixel fraction, chosen deterministically for a short demo—not a scientifically validated reference.

All tiles come from this one slide, so this illustrates colour changes, **not correction of between-site staining variation or improved model performance**. Inspect both reference and outputs for lost detail, clipping and unusual hues. Blank tiles are excluded from fitting. In a predictive study, choose/fix the reference using training data only.

Change `REFERENCE_TILE_ID` to another retained tile or shorten `STAIN_METHODS` for a quicker demo. Per-method elapsed times are reported. Normalised previews are saved in a separate directory; the original tile exports remain untouched.

**Practical tip: replace the background with a common estimated colour before normalisation.** Use the reviewed tissue and child-hole contours to define tissue. Compute the mean RGB of non-tissue pixels in the original slide thumbnail, excluding CODAIT-flagged artefacts; round to an RGB integer triplet. Apply that **same slide-level mean colour** outside tissue in both reference and source tiles, then restore it after normalisation. We display the actual value and save its sampling mask. This replaces spatial background variation without assuming that the scanner background is pure white.

We compare **unmasked** inputs with **mean-background-filled** inputs for every method. Both paths use the same reference and source tile identities. The unmasked path includes original background pixels in fitting/transformation and applies no restoration. Generic functions in `src.image_utils`, `src.tiling` and `src.normalization` perform colour estimation, contour projection, background replacement and the matched comparisons; this notebook only configures them and displays/saves results.

A contour is an estimate, not ground truth. Thumbnail boundaries are coarse, small holes may have been filled, and artefacts inside retained tissue remain. The estimated mean can also be biased by misclassified tissue or unflagged stitching artefacts. Inspect both the sampling mask and tile borders before research use. **Constant-background filling is not masked-statistics normalisation:** TIA Reinhard includes background pixels in its colour moments; stain-concentration percentiles also use the full image. Tissue proportion can still affect results. In a multi-slide study, choose and freeze a shared background convention with the reference rather than silently changing it between slides.


In [ ]:
STAIN_METHODS = ['reinhard', 'macenko', 'ruifrok', 'vahadane']
STAIN_SEED = 0  # The reusable comparison resets each call and restores caller RNG state.
retained = qc[qc.decision.eq('keep')].sort_values('tissue_fraction', ascending=False)
assert len(retained) >= 4, 'Need at least four retained tiles for this comparison; review QC first.'
REFERENCE_TILE_ID = retained.iloc[0].tile_id  # Edit to another retained tile ID if desired.
assert REFERENCE_TILE_ID in set(retained.tile_id)
source_ids = evenly_spaced(retained[retained.tile_id.ne(REFERENCE_TILE_ID)], 3).tile_id.tolist()
tissue_contours, tissue_holes = clam_contours(mask, (width, height))
normalisation_masks = {}
for tile_id in [REFERENCE_TILE_ID, *source_ids]:
    location = qc.set_index('tile_id').loc[tile_id, ['x_level0', 'y_level0']].to_numpy(dtype=int)
    normalisation_masks[tile_id] = contour_tile_mask(tissue_contours, tissue_holes,
        location, (read_w, read_h), TILE_PIXELS)
reference_rgb = np.asarray(tiles[REFERENCE_TILE_ID])
reference_mask = normalisation_masks[REFERENCE_TILE_ID]
# A single slide-level estimate is shared by the reference and every source tile.
background_sample_mask = ~mask & ~result['stages']['codait_excluded']
background_mean_rgb = mean_background_color(rgb, background_sample_mask)
BACKGROUND_RGB = np.rint(background_mean_rgb).astype(np.uint8)
print('Mean background RGB:', background_mean_rgb.round(2), '→ fill RGB:', BACKGROUND_RGB)
reference_prepared = prepare_stain_image(reference_rgb, tissue_mask=reference_mask,
                                        background_rgb=BACKGROUND_RGB)
stain_dir = OUT / 'stain_normalisation'
stain_dir.mkdir(exist_ok=True)
save_png(stain_dir / 'background_sample_mask.png', background_sample_mask)
save_png(stain_dir / 'reference_original.png', reference_rgb)
save_png(stain_dir / 'reference.png', reference_prepared)
save_png(stain_dir / 'reference_tissue_mask.png', reference_mask)
fig, axes = plt.subplots(1, 2, figsize=(6, 3), layout='constrained')
for ax, image, title in zip(axes, [reference_rgb, reference_prepared],
                            ['Reference original', 'Reference with mean background']):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')
fig.suptitle(REFERENCE_TILE_ID)
fig.savefig(OUT / '05a_stain_reference.png')
plt.show()

# Show the exact source inputs for both paths before comparing normalised outputs.
fig, axes = plt.subplots(2, len(source_ids), figsize=(9, 6), layout='constrained')
for col, tile_id in enumerate(source_ids):
    original = np.asarray(tiles[tile_id])
    tissue = normalisation_masks[tile_id]
    prepared = prepare_stain_image(original, tissue_mask=tissue, background_rgb=BACKGROUND_RGB)
    assert np.array_equal(prepared[tissue], original[tissue])
    assert (prepared[~tissue] == BACKGROUND_RGB).all()
    save_png(stain_dir / f'{tile_id}_mean_background.png', prepared)
    save_png(stain_dir / f'{tile_id}_tissue_mask.png', tissue)
    axes[0, col].imshow(original)
    axes[0, col].set_title(f'{tile_id} · original input')
    axes[1, col].imshow(prepared)
    axes[1, col].set_title(f'{tile_id} · mean-background input')
for ax in axes.flat:
    ax.axis('off')
fig.savefig(OUT / '05b_stain_inputs.png')
plt.show()

source_images = {tile_id: np.asarray(tiles[tile_id]) for tile_id in source_ids}
source_masks = {tile_id: normalisation_masks[tile_id] for tile_id in source_ids}
original_snapshots = {key: value.copy() for key, value in source_images.items()}
reference_snapshot = reference_rgb.copy()
rng_before = np.random.get_state()
comparison = compare_stain_normalization(reference_rgb, source_images,
    reference_mask=reference_mask, source_masks=source_masks, background_rgb=BACKGROUND_RGB,
    methods=STAIN_METHODS, random_seed=STAIN_SEED, max_pixels=MAX_TILE_READ_PIXELS)
rng_after = np.random.get_state()
assert rng_before[0] == rng_after[0] and np.array_equal(rng_before[1], rng_after[1])
assert rng_before[2:] == rng_after[2:], 'Comparison changed the caller NumPy RNG state'
assert np.array_equal(reference_rgb, reference_snapshot), 'Reference was modified'
for method, outputs in comparison['images'].items():
    fig, axes = plt.subplots(3, len(source_ids), figsize=(9, 9), layout='constrained')
    for col, tile_id in enumerate(source_ids):
        original = source_images[tile_id]
        tissue = source_masks[tile_id]
        assert np.array_equal(original, original_snapshots[tile_id]), 'Original was modified'
        axes[0, col].imshow(original)
        axes[0, col].set_title(f'{tile_id} · original')
        for row, (mode, images) in enumerate(outputs.items(), 1):
            transformed = images[tile_id]
            assert transformed.dtype == np.uint8 and transformed.shape == original.shape
            if mode == 'constant_background':
                assert (transformed[~tissue] == BACKGROUND_RGB).all()
            axes[row, col].imshow(transformed)
            axes[row, col].set_title(f"{tile_id} · {'original background' if mode == 'unmasked' else 'mean background'}")
            save_png(stain_dir / f'{tile_id}_{method}_{mode}.png', transformed)
    for ax in axes.flat:
        ax.axis('off')
    fig.suptitle(f'TIA {method.title()} · same reference and source tiles for both paths')
    fig.savefig(OUT / f'05_stain_{method}_comparison.png')
    plt.show()
display(pd.DataFrame(comparison['timings']).round(2))

stain_provenance = {'backend': 'TIAToolbox', 'version': version('tiatoolbox'),
    'methods': STAIN_METHODS, 'numpy_seed_per_call': STAIN_SEED,
    'timings': comparison['timings'],
    'vahadane_caveat': 'TIA warns of instability with newer scikit-learn dictionary learning',
    'reference_tile_id': REFERENCE_TILE_ID,
    'reference_slide_id': SLIDE_ID, 'source_tile_ids': source_ids,
    'background_modes': {'unmasked': 'original reference/source RGB; no masking or restoration',
        'constant_background': 'replace reference/source background with shared slide mean before TIA; restore after'},
    'background_rgb': BACKGROUND_RGB.tolist(),
    'background_mean_rgb_unrounded': background_mean_rgb.tolist(),
    'background_estimation': 'mean of original thumbnail pixels outside final mask, excluding CODAIT candidates',
    'background_sample_pixels': int(background_sample_mask.sum()),
    'mask_source': 'CLAM level-0 contours and child holes from final combined thumbnail mask',
    'tissue_fractions': {key: float(value.mean()) for key, value in normalisation_masks.items()},
    'statistics_scope': 'TIA defaults in both paths; constant filling does not exclude background pixels from statistics',
    'reference_selection': 'highest coloured-pixel fraction among retained sampled tiles',
    'scope': 'same-slide visual demonstration only; raw exports unchanged'}
(stain_dir / 'provenance.json').write_text(json.dumps(stain_provenance, indent=2))


### 6. Export the evidence, not just the pictures
Save the mask, all sampled tiles (including excluded controls for audit), a coordinate/QC table and the settings. Downstream users must explicitly select `decision == "keep"`, then complete visual/artefact QC.

Level-0 coordinates, source region dimensions and effective output MPP allow each tile to be located again. Split machine-learning datasets **by patient**, keeping slides and tiles from one patient together.

In [ ]:
(OUT / 'tiles').mkdir(exist_ok=True)
for tile_id, tile in tiles.items():
    tile.save(OUT / 'tiles' / f'{tile_id}.png')
thumbnail.save(OUT / 'thumbnail.png')
for name, array in result['stages'].items():
    save_png(OUT / f'stage_{name}.png', array)
save_png(OUT / 'tissue_mask.png', mask)
qc['slide_id'] = SLIDE_ID
qc['case_id'] = source['case_submitter_id']
qc['read_level'] = 0
qc['read_width_level0'] = read_w
qc['read_height_level0'] = read_h
qc['output_width_px'] = qc['output_height_px'] = TILE_PIXELS
qc['output_mpp_x'] = actual_mpp_x
qc['output_mpp_y'] = actual_mpp_y
qc['tile_path'] = 'tiles/' + qc.tile_id + '.png'
qc.to_csv(OUT / 'tiles.csv', index=False)

provenance = {
    'source_path': str(SLIDE_PATH), 'openslide_metadata': slide_metadata,
    'tiling_backend': 'CLAM', 'clam_commit': CLAM_COMMIT, 'clam_contour_fn': CLAM_CONTOUR_FN,
    'normalisation_demo': stain_provenance, 'source_manifest': str(manifest_path),
    'source_file_id': SLIDE_ID, 'source_original_filename': source['file_name'],
    'source_md5_verified': source_md5, 'source_size_bytes': SLIDE_PATH.stat().st_size,
    'source_url': f'https://portal.gdc.cancer.gov/files/{SLIDE_ID}',
    'case_id': source['case_submitter_id'], 'project': source['project'],
    'run_utc': run_id, 'level_dimensions': levels, 'level_downsamples': downsamples,
    'native_mpp_xy': [mpp_x, mpp_y], 'thumbnail_level': thumb_level,
    'thumbnail_dimensions': [thumb_w, thumb_h], 'thumbnail_scale_xy': [scale_x, scale_y],
    'parameters': {'target_mpp': TARGET_MPP, 'tile_pixels': TILE_PIXELS,
        'min_tissue_fraction': MIN_TISSUE_FRACTION, 'masking': MASKING, 'artifacts': ARTIFACTS,
        'thumbnail_edge': THUMBNAIL_EDGE, 'max_thumbnail_read_pixels': MAX_THUMBNAIL_READ_PIXELS,
        'n_tissue_candidates': N_TISSUE_CANDIDATES, 'n_low_tissue_controls': N_LOW_TISSUE_CONTROLS,
        'max_grid_positions': MAX_GRID_POSITIONS, 'max_contours': MAX_CONTOURS, 'max_tile_read_pixels': MAX_TILE_READ_PIXELS,
        'max_tile_seconds': MAX_TILE_SECONDS},
    'preprocessing': result['metadata'],
    'colour_handling': 'raw tile exports retain decoded scanner RGB; optional TIA comparisons are saved separately; no ICC conversion',
    'sampling': 'evenly spaced within two coverage strata of upstream CLAM coordinates; not representative',
    'qc_scope': 'thumbnail CODAIT/TIA/CLAM QC plus raw tile colour coverage; focus remains visual review',
    'grid_positions': len(grid), 'tiles_read': len(qc),
    'decision_counts': {str(k): int(v) for k, v in qc.decision.value_counts().items()},
    'python': platform.python_version(),
    'source_module_sha256': {name: hashlib.sha256((PROJECT / 'src' / name).read_bytes()).hexdigest()
                             for name in ['inspection.py', 'image_utils.py', 'masking.py',
                                          'filters.py', 'tiling.py', 'quality.py', 'pipeline.py', 'normalization.py']},
    'versions': {name: version(name) for name in
                 ['openslide-python', 'tiatoolbox', 'numpy', 'pandas', 'Pillow', 'opencv-python', 'matplotlib']},
}
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print(f'Saved {len(qc)} tiles, coordinates, QC decisions, provenance, stage images and teaching figures.')

## Checks
These checks exercise the **real slide and saved outputs**: source identity, tile count, coordinate bounds, saved image dimensions and QC consistency. They verify execution and bookkeeping, **not biological accuracy or diagnostic suitability**.

In [ ]:
saved = pd.read_csv(OUT / 'tiles.csv')
assert len(saved) == len(tiles) <= N_TISSUE_CANDIDATES + N_LOW_TISSUE_CONTROLS
assert not saved.duplicated(['x_level0', 'y_level0']).any()
assert saved.x_level0.ge(0).all() and saved.y_level0.ge(0).all()
assert (saved.x_level0 + read_w <= width).all()
assert (saved.y_level0 + read_h <= height).all()
assert saved.tissue_fraction.between(0, 1).all()
assert (saved.decision.eq('keep') == (saved.tissue_fraction.ge(MIN_TISSUE_FRACTION) &
                                          saved.thumbnail_tissue_fraction.ge(MIN_TISSUE_FRACTION))).all()
for row in saved.itertuples():
    with Image.open(OUT / row.tile_path) as image:
        assert image.size == (TILE_PIXELS, TILE_PIXELS)
        assert np.array_equal(np.asarray(image), np.asarray(tiles[row.tile_id]))
assert json.loads((OUT / 'provenance.json').read_text())['source_md5_verified'] == source_md5
print(f'Real-slide checks passed in {time.monotonic() - started:.1f} seconds.')
print(f'{len(saved)} sampled tiles: {(saved.decision == "keep").sum()} pass coverage; '
      f'{(saved.decision == "exclude").sum()} fail coverage. No focus/artefact approval is implied.')
print(f'Inspect saved results at: {OUT}')

## Next steps · questions for the group
1. Change `MASKING["max_hole_area"]` to 0, or `MASKING["median_size"]` to 3. Inspect which structures change. Then change `MIN_TISSUE_FRACTION` from 0.60 to 0.80 and rerun. Which lung regions are disproportionately lost? Remember that this also changes sampling strata.
2. Why might a lower Laplacian variance reflect tissue morphology rather than blur?
3. What changes when you double MPP but keep 256 × 256 pixels? **The physical field of view doubles in each direction.**
4. Why would randomly splitting tiles from one patient between training and test sets overstate performance?

For a real study, review mask failures across slides and tissue types, calibrate focus and artefact rules, choose scale for the biological question, and evaluate optional stain handling on held-out patients. This single-slide demonstration does not establish those choices.

### References and source
- [CLAM segmentation implementation](https://github.com/mahmoodlab/CLAM/blob/master/wsi_core/WholeSlideImage.py): saturation/median/threshold/closing and contour-based hole selection. Our bounded raster hole policy differs.
- [TIA OtsuTissueMasker](https://tia-toolbox.readthedocs.io/en/latest/_autosummary/tiatoolbox.tools.tissuemask.OtsuTissueMasker.html): grayscale thresholding.
- CODAIT filter attribution and original rule constants are retained in `src/filters.py` and `third_party/`.

- [OpenSlide Python documentation](https://openslide.org/api/python/): pyramid levels, physical metadata and region coordinates.
- [histolab tissue-mask examples](https://histolab.readthedocs.io/en/latest/tissue_masks.html): mask overlays and multiple tissue fragments.
- [Wölflein et al., Benchmarking Pathology Feature Extractors](https://arxiv.org/abs/2311.11772): task-specific evidence concerning stain normalisation; findings do not establish a universal rule.
- [GDC source record](https://portal.gdc.cancer.gov/files/20c9afac-0d0c-4fcf-b3bf-a2beba3dd76b): local public TCGA-LUAD WSI, identified by the project manifest and verified checksum. The WSI is not bundled into the notebook.
- [TIA stain-normalisation API](https://tia-toolbox.readthedocs.io/en/latest/_autosummary/tiatoolbox.tools.stainnorm.html): actual Reinhard, Macenko, Ruifrok and Vahadane implementations used above.
